In [2]:
from pathlib import Path
from collections import defaultdict
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Random Seed
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# Paths
# 当前 notebook 位于 taobao-recsys/notebooks/
# ============================================================

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "dev"

TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
VALIDATION_GT_PATH = DATA_DIR / "validation_ground_truth.csv"

TEST_PATH = DATA_DIR / "test.csv"
TEST_GT_PATH = DATA_DIR / "test_ground_truth.csv"


print("Train path:", TRAIN_PATH)
print("Validation path:", VALIDATION_PATH)
print("Validation GT path:", VALIDATION_GT_PATH)

print("\nCUDA available:", torch.cuda.is_available())

Train path: ..\data\processed\dev\train.csv
Validation path: ..\data\processed\dev\validation.csv
Validation GT path: ..\data\processed\dev\validation_ground_truth.csv

CUDA available: False


In [3]:
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
validation_gt = pd.read_csv(VALIDATION_GT_PATH)

print("=" * 60)
print("TRAIN")
print("=" * 60)

print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

display(train_df.head())


print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)

print("Shape:", validation_df.shape)
print("Columns:", validation_df.columns.tolist())

display(validation_df.head())


print("\n" + "=" * 60)
print("VALIDATION GROUND TRUTH")
print("=" * 60)

print("Shape:", validation_gt.shape)
print("Columns:", validation_gt.columns.tolist())

display(validation_gt.head())

TRAIN
Shape: (724885, 5)
Columns: ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']


,user_id,item_id,category_id,behavior_type,timestamp
0,100,4572582,2188684,pv,1511550516
1,100,2971043,4869428,pv,1511550600
2,100,2379198,4869428,pv,1511550806
3,100,2971043,4869428,pv,1511550885
4,100,1220136,4869428,pv,1511550908



VALIDATION
Shape: (138406, 5)
Columns: ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']


,user_id,item_id,category_id,behavior_type,timestamp
0,100,4140845,2096639,pv,1512173626
1,100,3656511,3312150,pv,1512177480
2,100,2772937,3114694,pv,1512192261
3,100,2859169,2096639,pv,1512201298
4,1000,4599135,245312,pv,1512228047



VALIDATION GROUND TRUTH
Shape: (2540, 2)
Columns: ['user_id', 'item_id']


,user_id,item_id
0,100600,4759580
1,100800,4558727
2,1008100,2700676
3,1011100,1845719
4,1012000,264220


In [4]:
# ============================================================
# Cell 3
# Prepare Positive Interactions + ID Mapping
# ============================================================

# 只使用 pv 行为作为当前 Two-Tower 的正反馈
train_pv = train_df[
    train_df["behavior_type"] == "pv"
].copy()

validation_pv = validation_df[
    validation_df["behavior_type"] == "pv"
].copy()


# 同一个 user-item 可能被点击多次
# Two-Tower baseline 中先只保留一份正样本
train_pv = (
    train_pv[
        ["user_id", "item_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

validation_pv = (
    validation_pv[
        ["user_id", "item_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


# ============================================================
# 使用 Train 构建 user / item vocabulary
# ============================================================

unique_users = train_pv["user_id"].unique()
unique_items = train_pv["item_id"].unique()


user2idx = {
    user_id: idx
    for idx, user_id in enumerate(unique_users)
}

item2idx = {
    item_id: idx
    for idx, item_id in enumerate(unique_items)
}


idx2user = {
    idx: user_id
    for user_id, idx in user2idx.items()
}

idx2item = {
    idx: item_id
    for item_id, idx in item2idx.items()
}


num_users = len(user2idx)
num_items = len(item2idx)


print("Train positive pairs:", len(train_pv))
print("Validation positive pairs:", len(validation_pv))

print("\nNumber of train users:", num_users)
print("Number of train items:", num_items)

print("\nExample user mapping:")
print(list(user2idx.items())[:5])

print("\nExample item mapping:")
print(list(item2idx.items())[:5])

Train positive pairs: 521287
Validation positive pairs: 107337

Number of train users: 9836
Number of train items: 302016

Example user mapping:
[(np.int64(100), 0), (np.int64(1000), 1), (np.int64(1001000), 2), (np.int64(1001500), 3), (np.int64(1001700), 4)]

Example item mapping:
[(np.int64(4572582), 0), (np.int64(2971043), 1), (np.int64(2379198), 2), (np.int64(1220136), 3), (np.int64(2518420), 4)]


In [5]:
# ============================================================
# Cell 4
# Map Original IDs -> Continuous Indices
# ============================================================

train_pv["user_idx"] = (
    train_pv["user_id"]
    .map(user2idx)
)

train_pv["item_idx"] = (
    train_pv["item_id"]
    .map(item2idx)
)


# Validation 中可能存在训练阶段从未出现过的 user / item
# 当前 baseline 暂时无法给它们查 Embedding
validation_seen = validation_pv[
    validation_pv["user_id"].isin(user2idx)
    &
    validation_pv["item_id"].isin(item2idx)
].copy()


validation_seen["user_idx"] = (
    validation_seen["user_id"]
    .map(user2idx)
)

validation_seen["item_idx"] = (
    validation_seen["item_id"]
    .map(item2idx)
)


# 转成 int
train_pv["user_idx"] = train_pv["user_idx"].astype(int)
train_pv["item_idx"] = train_pv["item_idx"].astype(int)

validation_seen["user_idx"] = (
    validation_seen["user_idx"].astype(int)
)

validation_seen["item_idx"] = (
    validation_seen["item_idx"].astype(int)
)


print("Train pairs:", len(train_pv))

print(
    "Validation pairs before filtering:",
    len(validation_pv)
)

print(
    "Validation pairs after seen filtering:",
    len(validation_seen)
)

print(
    "Removed cold-start pairs:",
    len(validation_pv) - len(validation_seen)
)


display(train_pv.head())
display(validation_seen.head())

Train pairs: 521287
Validation pairs before filtering: 107337
Validation pairs after seen filtering: 64148
Removed cold-start pairs: 43189


,user_id,item_id,user_idx,item_idx
0,100,4572582,0,0
1,100,2971043,0,1
2,100,2379198,0,2
3,100,1220136,0,3
4,100,2518420,0,4


,user_id,item_id,user_idx,item_idx
3,100,2859169,0,27378
4,1000,4599135,1,94447
5,1000,3532002,1,216289
12,1001000,4464239,2,103
13,1001000,2063954,2,59268


In [6]:
# ============================================================
# Cell 5
# Build user -> positive item set
# ============================================================

user_positive_items = defaultdict(set)

for row in train_pv.itertuples(index=False):

    user_positive_items[
        row.user_idx
    ].add(
        row.item_idx
    )


print(
    "Users in positive-item dictionary:",
    len(user_positive_items)
)


# 看一个用户作为检查
example_user = next(
    iter(user_positive_items)
)

print(
    "\nExample user_idx:",
    example_user
)

print(
    "Number of positive items:",
    len(
        user_positive_items[
            example_user
        ]
    )
)

print(
    "First 10 positive item_idx:",
    list(
        user_positive_items[
            example_user
        ]
    )[:10]
)

Users in positive-item dictionary: 9836

Example user_idx: 0
Number of positive items: 50
First 10 positive item_idx: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [7]:
# ============================================================
# Cell 6
# Build Train / Validation Interactions
# ============================================================

train_interactions = list(
    zip(
        train_pv["user_idx"].tolist(),
        train_pv["item_idx"].tolist(),
    )
)

validation_interactions = list(
    zip(
        validation_seen["user_idx"].tolist(),
        validation_seen["item_idx"].tolist(),
    )
)


print("Train interactions:", len(train_interactions))

print(
    "Validation interactions:",
    len(validation_interactions)
)

print("\nFirst 5 train interactions:")
print(train_interactions[:5])

print("\nFirst 5 validation interactions:")
print(validation_interactions[:5])

Train interactions: 521287
Validation interactions: 64148

First 5 train interactions:
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]

First 5 validation interactions:
[(0, 27378), (1, 94447), (1, 216289), (2, 103), (2, 59268)]


In [8]:
# ============================================================
# Cell 7
# Negative Sampling + Dataset
# ============================================================

class TwoTowerDataset(Dataset):

    def __init__(
        self,
        interactions,
        user_positive_items,
        num_items,
        num_negatives=4,
    ):

        self.interactions = interactions

        self.user_positive_items = (
            user_positive_items
        )

        self.num_items = num_items

        self.num_negatives = (
            num_negatives
        )


    def __len__(self):

        return len(
            self.interactions
        )


    def sample_negative(
        self,
        user_idx,
        positive_item_idx,
    ):

        positive_items = (
            self.user_positive_items[
                user_idx
            ]
        )

        while True:

            negative_item_idx = (
                random.randint(
                    0,
                    self.num_items - 1,
                )
            )

            # 不能采到用户 Train 中已经点击过的商品
            # 同时不能等于当前正样本
            if (
                negative_item_idx
                not in positive_items
                and
                negative_item_idx
                != positive_item_idx
            ):

                return (
                    negative_item_idx
                )


    def __getitem__(
        self,
        idx,
    ):

        (
            user_idx,
            positive_item_idx,
        ) = self.interactions[idx]


        negative_item_indices = [

            self.sample_negative(
                user_idx,
                positive_item_idx,
            )

            for _ in range(
                self.num_negatives
            )
        ]


        return {

            "user": torch.tensor(
                user_idx,
                dtype=torch.long,
            ),

            "positive_item": torch.tensor(
                positive_item_idx,
                dtype=torch.long,
            ),

            "negative_items": torch.tensor(
                negative_item_indices,
                dtype=torch.long,
            ),
        }

In [9]:
# ============================================================
# Cell 8
# Create Dataset
# ============================================================

NUM_NEGATIVES = 4


train_dataset = TwoTowerDataset(
    interactions=train_interactions,
    user_positive_items=user_positive_items,
    num_items=num_items,
    num_negatives=NUM_NEGATIVES,
)


validation_dataset = TwoTowerDataset(
    interactions=validation_interactions,
    user_positive_items=user_positive_items,
    num_items=num_items,
    num_negatives=NUM_NEGATIVES,
)


print(
    "Train dataset size:",
    len(train_dataset)
)

print(
    "Validation dataset size:",
    len(validation_dataset)
)


# 检查一个样本
example_sample = train_dataset[0]

print("\nExample sample:")

print(example_sample)

Train dataset size: 521287
Validation dataset size: 64148

Example sample:
{'user': tensor(0), 'positive_item': tensor(0), 'negative_items': tensor([ 58369,  13112, 144194, 128393])}


In [10]:
# ============================================================
# Cell 9
# DataLoader
# ============================================================

BATCH_SIZE = 1024


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

Train batches: 510
Validation batches: 63


In [11]:
# ============================================================
# Cell 10
# Inspect One Batch
# ============================================================

batch = next(
    iter(train_loader)
)


print(
    "User shape:",
    batch["user"].shape
)

print(
    "Positive item shape:",
    batch["positive_item"].shape
)

print(
    "Negative items shape:",
    batch["negative_items"].shape
)


print("\nFirst 5 users:")
print(
    batch["user"][:5]
)


print("\nFirst 5 positive items:")
print(
    batch["positive_item"][:5]
)


print("\nFirst 5 negative-item rows:")
print(
    batch["negative_items"][:5]
)

User shape: torch.Size([1024])
Positive item shape: torch.Size([1024])
Negative items shape: torch.Size([1024, 4])

First 5 users:
tensor([ 435, 5912, 5176, 7404, 9420])

First 5 positive items:
tensor([ 16665, 138398, 187420, 246296, 292847])

First 5 negative-item rows:
tensor([[117026,  73158,  53736, 285929],
        [ 45580, 221208,  16663,  15622],
        [ 49123, 114629, 121981, 264951],
        [ 13912, 294254, 104248, 285706],
        [219949, 115574, 235514, 145852]])


In [12]:
# ============================================================
# Cell 11
# Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


In [13]:
# ============================================================
# Cell 12
# User Tower
# ============================================================

class UserTower(nn.Module):

    def __init__(
        self,
        num_users,
        embedding_dim=64,
        hidden_dim=128,
        output_dim=64,
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim,
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                embedding_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
        )


    def forward(
        self,
        user_ids,
    ):

        user_embedding = (
            self.user_embedding(
                user_ids
            )
        )

        user_vector = (
            self.mlp(
                user_embedding
            )
        )

        user_vector = F.normalize(
            user_vector,
            p=2,
            dim=-1,
        )

        return user_vector

In [14]:
# ============================================================
# Cell 13
# Item Tower
# ============================================================

class ItemTower(nn.Module):

    def __init__(
        self,
        num_items,
        embedding_dim=64,
        hidden_dim=128,
        output_dim=64,
    ):
        super().__init__()

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim,
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                embedding_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
        )


    def forward(
        self,
        item_ids,
    ):

        item_embedding = (
            self.item_embedding(
                item_ids
            )
        )

        item_vector = (
            self.mlp(
                item_embedding
            )
        )

        item_vector = F.normalize(
            item_vector,
            p=2,
            dim=-1,
        )

        return item_vector

In [15]:
# ============================================================
# Cell 14
# Two-Tower Model
# ============================================================

class TwoTowerModel(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=64,
        hidden_dim=128,
        output_dim=64,
        temperature=0.1,
    ):
        super().__init__()

        self.user_tower = UserTower(
            num_users=num_users,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            output_dim=output_dim,
        )

        self.item_tower = ItemTower(
            num_items=num_items,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            output_dim=output_dim,
        )

        self.temperature = temperature


    def encode_users(
        self,
        user_ids,
    ):

        return self.user_tower(
            user_ids
        )


    def encode_items(
        self,
        item_ids,
    ):

        return self.item_tower(
            item_ids
        )


    def forward(
        self,
        user_ids,
        positive_item_ids,
        negative_item_ids,
    ):

        user_vectors = (
            self.encode_users(
                user_ids
            )
        )

        positive_item_vectors = (
            self.encode_items(
                positive_item_ids
            )
        )

        negative_item_vectors = (
            self.encode_items(
                negative_item_ids
            )
        )


        positive_logits = torch.sum(
            user_vectors
            *
            positive_item_vectors,
            dim=-1,
        )

        negative_logits = torch.sum(
            user_vectors.unsqueeze(1)
            *
            negative_item_vectors,
            dim=-1,
        )


        positive_logits = (
            positive_logits
            /
            self.temperature
        )

        negative_logits = (
            negative_logits
            /
            self.temperature
        )


        return (
            positive_logits,
            negative_logits,
        )

In [16]:
# ============================================================
# Cell 15
# Create Model
# ============================================================

EMBEDDING_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 64
TEMPERATURE = 0.1


model = TwoTowerModel(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    temperature=TEMPERATURE,
)


model = model.to(device)


print(model)

TwoTowerModel(
  (user_tower): UserTower(
    (user_embedding): Embedding(9836, 64)
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
    )
  )
  (item_tower): ItemTower(
    (item_embedding): Embedding(302016, 64)
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
    )
  )
)


In [17]:
# ============================================================
# Cell 16
# Forward Shape Check
# ============================================================

batch = next(
    iter(train_loader)
)


user_ids = (
    batch["user"]
    .to(device)
)

positive_item_ids = (
    batch["positive_item"]
    .to(device)
)

negative_item_ids = (
    batch["negative_items"]
    .to(device)
)


with torch.no_grad():

    (
        positive_logits,
        negative_logits,
    ) = model(
        user_ids,
        positive_item_ids,
        negative_item_ids,
    )


print(
    "Positive logits shape:",
    positive_logits.shape
)

print(
    "Negative logits shape:",
    negative_logits.shape
)

print(
    "\nFirst 5 positive logits:"
)

print(
    positive_logits[:5]
)

print(
    "\nFirst negative row:"
)

print(
    negative_logits[0]
)

Positive logits shape: torch.Size([1024])
Negative logits shape: torch.Size([1024, 4])

First 5 positive logits:
tensor([ 1.1663, -0.2844, -3.2147,  0.0753,  0.5183])

First negative row:
tensor([-0.1564,  0.1493,  0.3574,  1.8001])


In [18]:
# ============================================================
# Cell 17
# Loss + Optimizer
# ============================================================

LEARNING_RATE = 1e-3


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)


def calculate_loss(
    positive_logits,
    negative_logits,
):

    positive_labels = torch.ones_like(
        positive_logits
    )

    negative_labels = torch.zeros_like(
        negative_logits
    )


    positive_loss = (
        F.binary_cross_entropy_with_logits(
            positive_logits,
            positive_labels,
        )
    )


    negative_loss = (
        F.binary_cross_entropy_with_logits(
            negative_logits,
            negative_labels,
        )
    )


    loss = (
        positive_loss
        +
        negative_loss
    ) / 2


    return loss

In [19]:
loss = calculate_loss(
    positive_logits,
    negative_logits,
)

print(
    "Initial loss:",
    loss.item()
)

Initial loss: 0.8593322038650513


In [20]:
# ============================================================
# Cell 18
# Train One Epoch
# ============================================================

def train_one_epoch(
    model,
    data_loader,
    optimizer,
):

    model.train()

    total_loss = 0.0


    for batch in data_loader:

        user_ids = (
            batch["user"]
            .to(device)
        )

        positive_item_ids = (
            batch["positive_item"]
            .to(device)
        )

        negative_item_ids = (
            batch["negative_items"]
            .to(device)
        )


        optimizer.zero_grad()


        (
            positive_logits,
            negative_logits,
        ) = model(
            user_ids,
            positive_item_ids,
            negative_item_ids,
        )


        loss = calculate_loss(
            positive_logits,
            negative_logits,
        )


        loss.backward()


        optimizer.step()


        total_loss += (
            loss.item()
        )


    average_loss = (
        total_loss
        /
        len(data_loader)
    )


    return average_loss

In [21]:
# ============================================================
# Cell 19
# Validation
# ============================================================

def evaluate_loss(
    model,
    data_loader,
):

    model.eval()

    total_loss = 0.0


    with torch.no_grad():

        for batch in data_loader:

            user_ids = (
                batch["user"]
                .to(device)
            )

            positive_item_ids = (
                batch["positive_item"]
                .to(device)
            )

            negative_item_ids = (
                batch["negative_items"]
                .to(device)
            )


            (
                positive_logits,
                negative_logits,
            ) = model(
                user_ids,
                positive_item_ids,
                negative_item_ids,
            )


            loss = calculate_loss(
                positive_logits,
                negative_logits,
            )


            total_loss += (
                loss.item()
            )


    average_loss = (
        total_loss
        /
        len(data_loader)
    )


    return average_loss

In [22]:
# ============================================================
# Cell 20
# Training Loop + Save Best Model
# ============================================================

NUM_EPOCHS = 3


OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "artifacts"
    /
    "two_tower"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


BEST_MODEL_PATH = (
    OUTPUT_DIR
    /
    "two_tower_best.pt"
)


best_validation_loss = float(
    "inf"
)


for epoch in range(
    1,
    NUM_EPOCHS + 1,
):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
    )


    validation_loss = evaluate_loss(
        model,
        validation_loader,
    )


    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Validation Loss: {validation_loss:.6f}"
    )


    if (
        validation_loss
        <
        best_validation_loss
    ):

        best_validation_loss = (
            validation_loss
        )


        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "num_users":
                    num_users,

                "num_items":
                    num_items,

                "embedding_dim":
                    EMBEDDING_DIM,

                "hidden_dim":
                    HIDDEN_DIM,

                "output_dim":
                    OUTPUT_DIM,

                "temperature":
                    TEMPERATURE,
            },
            BEST_MODEL_PATH,
        )


        print(
            "Best model saved."
        )


print(
    "\nTraining finished."
)

print(
    "Best validation loss:",
    best_validation_loss
)

print(
    "Saved to:",
    BEST_MODEL_PATH
)

Epoch 01 | Train Loss: 0.698493 | Validation Loss: 0.692557
Best model saved.
Epoch 02 | Train Loss: 0.690275 | Validation Loss: 0.682466
Best model saved.
Epoch 03 | Train Loss: 0.679218 | Validation Loss: 0.660640
Best model saved.

Training finished.
Best validation loss: 0.6606400249496339
Saved to: ..\artifacts\two_tower\two_tower_best.pt


# Two-Tower Retrieval V2

本节从现有训练数据恢复与训练时完全一致的 `user_id/item_id -> embedding index` 映射，验证 checkpoint vocabulary 后持久化映射。随后只读加载最佳 checkpoint，导出归一化 user/item embeddings，并在 validation purchase ground truth 的 warm-start 子集上执行分块 exact Top-K retrieval。

输入：`train.csv`、`validation_ground_truth.csv`、`two_tower_best.pt`。输出：mapping CSV/metadata、embedding NPY、warm-start coverage、Top-K 指标及真实用户 sanity check。已有训练 cell 不会在本节重跑。

In [1]:
# Recover the exact training vocabulary and load the existing checkpoint.
from pathlib import Path
import hashlib
import json
import math
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'dev'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'two_tower'
TRAIN_PATH = DATA_DIR / 'train.csv'
VALIDATION_GT_PATH = DATA_DIR / 'validation_ground_truth.csv'
CHECKPOINT_PATH = ARTIFACT_DIR / 'two_tower_best.pt'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)
validation_gt = pd.read_csv(VALIDATION_GT_PATH)
train_pv = (
    train_df.loc[train_df['behavior_type'].eq('pv'), ['user_id', 'item_id']]
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_users = train_pv['user_id'].unique()
unique_items = train_pv['item_id'].unique()
user2idx = {int(user_id): idx for idx, user_id in enumerate(unique_users)}
item2idx = {int(item_id): idx for idx, item_id in enumerate(unique_items)}
idx2user = {idx: int(user_id) for idx, user_id in enumerate(unique_users)}
idx2item = {idx: int(item_id) for idx, item_id in enumerate(unique_items)}

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
expected_users, expected_items = 9_836, 302_016
recovered_users, recovered_items = len(user2idx), len(item2idx)
checkpoint_users = int(checkpoint['num_users'])
checkpoint_items = int(checkpoint['num_items'])
if (recovered_users, recovered_items) != (expected_users, expected_items):
    raise RuntimeError(f'Recovered vocabulary mismatch: {(recovered_users, recovered_items)}')
if (checkpoint_users, checkpoint_items) != (expected_users, expected_items):
    raise RuntimeError(f'Checkpoint vocabulary mismatch: {(checkpoint_users, checkpoint_items)}')

class UserTower(nn.Module):
    def __init__(self, num_users, embedding_dim=64, hidden_dim=128, output_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.mlp = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))
    def forward(self, user_ids):
        return F.normalize(self.mlp(self.user_embedding(user_ids)), p=2, dim=-1)

class ItemTower(nn.Module):
    def __init__(self, num_items, embedding_dim=64, hidden_dim=128, output_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.mlp = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))
    def forward(self, item_ids):
        return F.normalize(self.mlp(self.item_embedding(item_ids)), p=2, dim=-1)

class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, hidden_dim=128, output_dim=64, temperature=0.1):
        super().__init__()
        self.user_tower = UserTower(num_users, embedding_dim, hidden_dim, output_dim)
        self.item_tower = ItemTower(num_items, embedding_dim, hidden_dim, output_dim)
        self.temperature = temperature
    def encode_users(self, user_ids):
        return self.user_tower(user_ids)
    def encode_items(self, item_ids):
        return self.item_tower(item_ids)

model = TwoTowerModel(
    checkpoint_users, checkpoint_items, checkpoint['embedding_dim'],
    checkpoint['hidden_dim'], checkpoint['output_dim'], checkpoint['temperature']
)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()

user_mapping = pd.DataFrame({'user_id': unique_users.astype('int64'), 'user_idx': np.arange(recovered_users)})
item_mapping = pd.DataFrame({'item_id': unique_items.astype('int64'), 'item_idx': np.arange(recovered_items)})
USER_MAPPING_PATH = ARTIFACT_DIR / 'user_mapping.csv'
ITEM_MAPPING_PATH = ARTIFACT_DIR / 'item_mapping.csv'
MAPPING_METADATA_PATH = ARTIFACT_DIR / 'mapping_metadata.json'
user_mapping.to_csv(USER_MAPPING_PATH, index=False)
item_mapping.to_csv(ITEM_MAPPING_PATH, index=False)
sha256 = hashlib.sha256()
with TRAIN_PATH.open('rb') as source_file:
    for block in iter(lambda: source_file.read(1024 * 1024), b''):
        sha256.update(block)
mapping_metadata = {
    'mapping_method': "filter behavior_type == 'pv'; select user_id,item_id; drop_duplicates keeping first occurrence; pandas unique order",
    'train_csv_sha256': sha256.hexdigest(),
    'num_users': recovered_users,
    'num_items': recovered_items,
    'checkpoint': CHECKPOINT_PATH.name,
}
MAPPING_METADATA_PATH.write_text(json.dumps(mapping_metadata, indent=2), encoding='utf-8')

print('Mapping method:', mapping_metadata['mapping_method'])
print('Recovered vocabulary:', recovered_users, 'users /', recovered_items, 'items')
print('Checkpoint vocabulary:', checkpoint_users, 'users /', checkpoint_items, 'items')
print('Strict state_dict load: OK')
print('Saved mappings:', USER_MAPPING_PATH.name, ITEM_MAPPING_PATH.name, MAPPING_METADATA_PATH.name)

Mapping method: filter behavior_type == 'pv'; select user_id,item_id; drop_duplicates keeping first occurrence; pandas unique order
Recovered vocabulary: 9836 users / 302016 items
Checkpoint vocabulary: 9836 users / 302016 items
Strict state_dict load: OK
Saved mappings: user_mapping.csv item_mapping.csv mapping_metadata.json


## Export normalized embeddings

对全部已见 user 和 candidate item 分批编码。输出为 `float32` NPY；tower 输出已经 L2-normalized，因此后续 inner product 等价于 cosine similarity。

In [2]:
@torch.inference_mode()
def encode_all(encoder, count, batch_size=8_192):
    parts = []
    for start in range(0, count, batch_size):
        indices = torch.arange(start, min(start + batch_size, count), dtype=torch.long)
        parts.append(encoder(indices).cpu().numpy().astype('float32', copy=False))
    return np.concatenate(parts, axis=0)

user_embeddings = encode_all(model.encode_users, recovered_users)
item_embeddings = encode_all(model.encode_items, recovered_items)
USER_EMBEDDINGS_PATH = ARTIFACT_DIR / 'user_embeddings.npy'
ITEM_EMBEDDINGS_PATH = ARTIFACT_DIR / 'item_embeddings.npy'
np.save(USER_EMBEDDINGS_PATH, user_embeddings)
np.save(ITEM_EMBEDDINGS_PATH, item_embeddings)

assert user_embeddings.shape == (9_836, 64)
assert item_embeddings.shape == (302_016, 64)
assert np.isfinite(user_embeddings).all() and np.isfinite(item_embeddings).all()
print('Embedding dimension:', item_embeddings.shape[1])
print('User embedding shape:', user_embeddings.shape)
print('Item embedding shape:', item_embeddings.shape)
print('Candidate item count:', len(item_embeddings))
print('Mean user L2 norm:', float(np.linalg.norm(user_embeddings, axis=1).mean()))
print('Mean item L2 norm:', float(np.linalg.norm(item_embeddings, axis=1).mean()))
print('Saved embeddings:', USER_EMBEDDINGS_PATH.name, ITEM_EMBEDDINGS_PATH.name)

Embedding dimension: 64
User embedding shape: (9836, 64)
Item embedding shape: (302016, 64)
Candidate item count: 302016
Mean user L2 norm: 1.0
Mean item L2 norm: 1.0
Saved embeddings: user_embeddings.npy item_embeddings.npy


## Warm-start evaluation set and chunked exact retrieval

Evaluation target 是 validation 当日的唯一 purchase pair。warm-start interaction 要求 user 和 target item 都存在于训练时 PV vocabulary；指标分母只使用这些可由当前 ID-only 模型表示的 warm targets。检索候选为全部 302,016 个 train-PV items，不过滤历史商品，以保持与现有 Popularity/ItemCF future-purchase protocol 一致。

Exact retrieval 对 user 和 item 分块，只保留各 item chunk 的 Top-50 并滚动合并，不构造 `[9836, 302016]` 完整 score matrix。先对 8 个真实用户做 sanity check，再运行完整 warm-start evaluation。

In [3]:
gt_total_users = validation_gt['user_id'].nunique()
gt_total_interactions = len(validation_gt)
user_seen_mask = validation_gt['user_id'].isin(user2idx)
item_seen_mask = validation_gt['item_id'].isin(item2idx)
warm_mask = user_seen_mask & item_seen_mask
warm_gt_df = validation_gt.loc[warm_mask, ['user_id', 'item_id']].copy()
warm_ground_truth = warm_gt_df.groupby('user_id')['item_id'].apply(set).to_dict()
warm_user_ids = np.array(sorted(warm_ground_truth), dtype=np.int64)
warm_user_indices = np.array([user2idx[int(user_id)] for user_id in warm_user_ids], dtype=np.int64)

print('GT total users:', gt_total_users)
print('GT total interactions:', gt_total_interactions)
print('Unseen users:', validation_gt.loc[~user_seen_mask, 'user_id'].nunique())
print('Unseen target item interactions:', int((~item_seen_mask).sum()))
print('Unseen target unique items:', validation_gt.loc[~item_seen_mask, 'item_id'].nunique())
print('Warm-start evaluation users:', len(warm_ground_truth))
print('Warm-start evaluation interactions:', len(warm_gt_df))
print('Warm-start user coverage:', len(warm_ground_truth) / gt_total_users)
print('Warm-start interaction coverage:', len(warm_gt_df) / gt_total_interactions)

def exact_chunked_topk(user_indices, top_k=50, user_batch_size=64, item_chunk_size=50_000):
    candidate_tensor = torch.from_numpy(item_embeddings)
    all_scores, all_indices = [], []
    for user_start in range(0, len(user_indices), user_batch_size):
        batch_indices = user_indices[user_start:user_start + user_batch_size]
        user_tensor = torch.from_numpy(user_embeddings[batch_indices])
        batch_size = len(batch_indices)
        best_scores = torch.full((batch_size, top_k), -torch.inf, dtype=torch.float32)
        best_indices = torch.full((batch_size, top_k), -1, dtype=torch.long)
        for item_start in range(0, len(item_embeddings), item_chunk_size):
            item_end = min(item_start + item_chunk_size, len(item_embeddings))
            score_chunk = user_tensor @ candidate_tensor[item_start:item_end].T
            local_k = min(top_k, item_end - item_start)
            chunk_scores, chunk_indices = torch.topk(score_chunk, k=local_k, dim=1)
            chunk_indices += item_start
            merged_scores = torch.cat([best_scores, chunk_scores], dim=1)
            merged_indices = torch.cat([best_indices, chunk_indices], dim=1)
            best_scores, positions = torch.topk(merged_scores, k=top_k, dim=1)
            best_indices = torch.gather(merged_indices, 1, positions)
        all_scores.append(best_scores.numpy())
        all_indices.append(best_indices.numpy())
    return np.vstack(all_scores), np.vstack(all_indices)

sanity_user_ids = warm_user_ids[:8]
sanity_user_indices = np.array([user2idx[int(user_id)] for user_id in sanity_user_ids], dtype=np.int64)
sanity_scores, sanity_item_indices = exact_chunked_topk(sanity_user_indices, top_k=10, user_batch_size=8)
sanity_dense_scores = torch.from_numpy(user_embeddings[sanity_user_indices]) @ torch.from_numpy(item_embeddings).T
_, sanity_dense_indices = torch.topk(sanity_dense_scores, k=10, dim=1)
chunked_matches_small_dense = np.array_equal(sanity_item_indices, sanity_dense_indices.numpy())
assert chunked_matches_small_dense
sanity_rows = []
for row, user_id in enumerate(sanity_user_ids):
    retrieved_ids = [idx2item[int(idx)] for idx in sanity_item_indices[row]]
    sanity_rows.append({'user_id': int(user_id), 'retrieved_top10': retrieved_ids, 'actual_purchase_gt': sorted(warm_ground_truth[int(user_id)])})
sanity_df = pd.DataFrame(sanity_rows)
assert sanity_item_indices.shape == (8, 10)
assert all(len(set(row)) == 10 for row in sanity_item_indices)
print('Sanity retrieval tensor shapes:', sanity_scores.shape, sanity_item_indices.shape)
print('Chunked Top-10 matches small dense exact calculation:', chunked_matches_small_dense)
display(sanity_df)
for record in sanity_rows:
    print(f"user_id={record['user_id']} | retrieved_top10={record['retrieved_top10']} | actual_purchase_gt={record['actual_purchase_gt']}")

GT total users: 1760
GT total interactions: 2540
Unseen users: 16
Unseen target item interactions: 1032
Unseen target unique items: 1023
Warm-start evaluation users: 1170
Warm-start evaluation interactions: 1496
Warm-start user coverage: 0.6647727272727273
Warm-start interaction coverage: 0.5889763779527559


Sanity retrieval tensor shapes: (8, 10) (8, 10)
Chunked Top-10 matches small dense exact calculation: True


,user_id,retrieved_top10,actual_purchase_gt
0,900,"[3845720, 2367945, 5128303, 619901, 3114069, 3...",[4601558]
1,1100,"[3845720, 2367945, 3371523, 2988047, 3715112, ...",[351376]
2,1200,"[3845720, 2367945, 5128303, 3371523, 619901, 2...",[1581301]
3,1400,"[3845720, 2367945, 3031354, 2988047, 1591862, ...",[2346309]
4,2900,"[3845720, 2367945, 5128303, 2453685, 3371523, ...",[1655134]
5,3300,"[2367945, 3845720, 2453685, 619901, 3114069, 1...",[2665565]
6,8600,"[3845720, 2367945, 3371523, 2453685, 1591862, ...",[1306004]
7,10200,"[3845720, 2367945, 3371523, 2364679, 3715112, ...","[2104146, 3011657]"


user_id=900 | retrieved_top10=[3845720, 2367945, 5128303, 619901, 3114069, 3715112, 2453685, 2988047, 1591862, 3371523] | actual_purchase_gt=[4601558]
user_id=1100 | retrieved_top10=[3845720, 2367945, 3371523, 2988047, 3715112, 3031354, 5128303, 2453685, 1951985, 2364679] | actual_purchase_gt=[351376]
user_id=1200 | retrieved_top10=[3845720, 2367945, 5128303, 3371523, 619901, 2453685, 3715112, 1591862, 2988047, 779670] | actual_purchase_gt=[1581301]
user_id=1400 | retrieved_top10=[3845720, 2367945, 3031354, 2988047, 1591862, 5128303, 223790, 619901, 2453685, 3209415] | actual_purchase_gt=[2346309]
user_id=2900 | retrieved_top10=[3845720, 2367945, 5128303, 2453685, 3371523, 619901, 3114069, 2988047, 3715112, 3031354] | actual_purchase_gt=[1655134]
user_id=3300 | retrieved_top10=[2367945, 3845720, 2453685, 619901, 3114069, 1154768, 5128303, 223790, 3715112, 3371523] | actual_purchase_gt=[2665565]
user_id=8600 | retrieved_top10=[3845720, 2367945, 3371523, 2453685, 1591862, 1951985, 298804

In [4]:
retrieval_started = time.perf_counter()
top50_scores, top50_item_indices = exact_chunked_topk(warm_user_indices, top_k=50)
exact_retrieval_seconds = time.perf_counter() - retrieval_started
two_tower_recommendations = {
    int(user_id): [idx2item[int(idx)] for idx in top50_item_indices[row]]
    for row, user_id in enumerate(warm_user_ids)
}

def recall_at_k(recommendations, ground_truth, k):
    recs, gt = set(recommendations[:k]), set(ground_truth)
    return 0.0 if not gt else len(recs & gt) / len(gt)

def hitrate_at_k(recommendations, ground_truth, k):
    recs, gt = set(recommendations[:k]), set(ground_truth)
    return 0.0 if not gt else float(bool(recs & gt))

def ndcg_at_k(recommendations, ground_truth, k):
    gt = set(ground_truth)
    if not gt:
        return 0.0
    dcg = sum(1.0 / math.log2(rank + 1) for rank, item in enumerate(recommendations[:k], 1) if item in gt)
    ideal_hits = min(len(gt), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return 0.0 if idcg == 0 else dcg / idcg

def evaluate_recommendations(recommendations, ground_truth, ks=(10, 20, 50)):
    rows = []
    for k in ks:
        recalls, hits, ndcgs = [], [], []
        for user_id, gt_items in ground_truth.items():
            recs = recommendations.get(int(user_id), [])
            recalls.append(recall_at_k(recs, gt_items, k))
            hits.append(hitrate_at_k(recs, gt_items, k))
            ndcgs.append(ndcg_at_k(recs, gt_items, k))
        rows.append({'K': k, 'Recall': np.mean(recalls), 'HitRate': np.mean(hits), 'NDCG': np.mean(ndcgs)})
    return pd.DataFrame(rows)

two_tower_results = evaluate_recommendations(two_tower_recommendations, warm_ground_truth)
print('Full exact Top-50 score/index shapes:', top50_scores.shape, top50_item_indices.shape)
print(f'Exact retrieval elapsed seconds: {exact_retrieval_seconds:.3f}')
display(two_tower_results)

sample_rng = np.random.default_rng(42)
sample_user_ids = sample_rng.choice(warm_user_ids, size=5, replace=False)
sample_rows = [
    {'user_id': int(user_id), 'retrieved_top10': two_tower_recommendations[int(user_id)][:10], 'actual_purchase_gt': sorted(warm_ground_truth[int(user_id)])}
    for user_id in sample_user_ids
]
display(pd.DataFrame(sample_rows))
for record in sample_rows:
    print(f"user_id={record['user_id']} | retrieved_top10={record['retrieved_top10']} | actual_purchase_gt={record['actual_purchase_gt']}")

Full exact Top-50 score/index shapes: (1170, 50) (1170, 50)
Exact retrieval elapsed seconds: 0.933


,K,Recall,HitRate,NDCG
0,10,0.002564,0.002564,0.000920
1,20,0.002991,0.003419,0.001066
2,50,0.003846,0.004274,0.001221


,user_id,retrieved_top10,actual_purchase_gt
0,770000,"[3845720, 3715112, 2367945, 3031354, 5128303, ...",[2522943]
1,412700,"[3845720, 2367945, 1444258, 779670, 2364679, 8...","[1871089, 3525595]"
2,635600,"[3845720, 2367945, 2453685, 5128303, 619901, 3...",[2699523]
3,81600,"[3845720, 2367945, 3715112, 3371523, 2364679, ...",[3281387]
4,402800,"[3845720, 2367945, 2988047, 2453685, 1951985, ...",[48995]


user_id=770000 | retrieved_top10=[3845720, 3715112, 2367945, 3031354, 5128303, 3371523, 2988047, 2588261, 779670, 587994] | actual_purchase_gt=[2522943]
user_id=412700 | retrieved_top10=[3845720, 2367945, 1444258, 779670, 2364679, 812879, 5128303, 3031354, 2867204, 587994] | actual_purchase_gt=[1871089, 3525595]
user_id=635600 | retrieved_top10=[3845720, 2367945, 2453685, 5128303, 619901, 3371523, 3031354, 4211339, 3114069, 2331370] | actual_purchase_gt=[2699523]
user_id=81600 | retrieved_top10=[3845720, 2367945, 3715112, 3371523, 2364679, 3174951, 1591862, 1951985, 3031354, 779670] | actual_purchase_gt=[3281387]
user_id=402800 | retrieved_top10=[3845720, 2367945, 2988047, 2453685, 1951985, 2364679, 5128303, 2588261, 223790, 3114069] | actual_purchase_gt=[48995]


## Fair comparison with existing recall baselines

原 `03` Notebook 的 Popularity/ItemCF 数值是在包含 cold targets 的完整 validation GT 上计算，不能直接与当前 warm-only Two-Tower 数值比较。这里复用相同的数据构造、推荐逻辑和 metric 定义，但让三个模型都在同一个 `warm_ground_truth` 上重新评估，得到公平对比。ItemCF 仍使用 train 全行为、每用户最近 50 个 unique items 和每个 seed Top-200 neighbors。

In [5]:
# Popularity on the identical warm-start purchase evaluation set.
popular_items = train_df.groupby('item_id').size().sort_values(ascending=False).index.astype('int64').tolist()
popularity_recommendations = {int(user_id): popular_items[:50] for user_id in warm_ground_truth}
popularity_warm_results = evaluate_recommendations(popularity_recommendations, warm_ground_truth)

# ItemCF construction matching notebook 03.
from scipy.sparse import csr_matrix
itemcf_train = (
    train_df.sort_values('timestamp')
    .drop_duplicates(subset=['user_id', 'item_id'], keep='last')
    .sort_values(['user_id', 'timestamp'])
    .groupby('user_id', group_keys=False)
    .tail(50)
)
itemcf_user_history = itemcf_train.groupby('user_id')['item_id'].apply(list).to_dict()
user_codes, itemcf_user_ids = pd.factorize(itemcf_train['user_id'], sort=True)
item_codes, itemcf_item_ids = pd.factorize(itemcf_train['item_id'], sort=True)
X = csr_matrix(
    (np.ones(len(itemcf_train), dtype=np.float32), (user_codes, item_codes)),
    shape=(len(itemcf_user_ids), len(itemcf_item_ids)),
)
item_to_col = {int(item_id): idx for idx, item_id in enumerate(itemcf_item_ids)}
seed_items = sorted({
    int(item)
    for user_id in warm_ground_truth
    for item in itemcf_user_history.get(user_id, [])
    if int(item) in item_to_col
})
seed_cols = np.array([item_to_col[item] for item in seed_items], dtype=np.int32)
item_user_counts = np.asarray(X.sum(axis=0)).ravel()
co_matrix = (X[:, seed_cols].T @ X).tocsr()
sim_matrix = {}
for seed_pos, seed_item in enumerate(seed_items):
    start, end = co_matrix.indptr[seed_pos], co_matrix.indptr[seed_pos + 1]
    candidate_cols = co_matrix.indices[start:end]
    co_counts = co_matrix.data[start:end]
    seed_col = seed_cols[seed_pos]
    keep = candidate_cols != seed_col
    candidate_cols, co_counts = candidate_cols[keep], co_counts[keep]
    if len(candidate_cols) == 0:
        sim_matrix[seed_item] = []
        continue
    similarities = co_counts / np.sqrt(item_user_counts[seed_col] * item_user_counts[candidate_cols])
    if len(similarities) > 200:
        positions = np.argpartition(similarities, -200)[-200:]
        positions = positions[np.argsort(similarities[positions])[::-1]]
    else:
        positions = np.argsort(similarities)[::-1]
    sim_matrix[seed_item] = [(int(itemcf_item_ids[candidate_cols[pos]]), float(similarities[pos])) for pos in positions]

def recommend_itemcf(user_id, k=50):
    rank = defaultdict(float)
    for hist_item in itemcf_user_history.get(user_id, []):
        for candidate_item, similarity in sim_matrix.get(int(hist_item), []):
            rank[candidate_item] += similarity
    return [item for item, _ in sorted(rank.items(), key=lambda pair: pair[1], reverse=True)[:k]]

itemcf_recommendations = {int(user_id): recommend_itemcf(int(user_id), 50) for user_id in warm_ground_truth}
itemcf_warm_results = evaluate_recommendations(itemcf_recommendations, warm_ground_truth)
comparison_results = pd.concat([
    popularity_warm_results.assign(Model='Popularity'),
    itemcf_warm_results.assign(Model='ItemCF'),
    two_tower_results.assign(Model='Two-Tower'),
]) [['Model', 'K', 'Recall', 'HitRate', 'NDCG']].reset_index(drop=True)
print('Fair evaluation set: warm-start validation purchase targets')
print('Evaluation users/interactions:', len(warm_ground_truth), '/', len(warm_gt_df))
display(comparison_results)

Fair evaluation set: warm-start validation purchase targets
Evaluation users/interactions: 1170 / 1496


,Model,K,Recall,HitRate,NDCG
0,Popularity,10,0.002991,0.003419,0.000992
1,Popularity,20,0.004701,0.005128,0.001405
2,Popularity,50,0.007123,0.010256,0.002016
3,ItemCF,10,0.108468,0.129060,0.051385
4,ItemCF,20,0.189158,0.217094,0.072717
5,ItemCF,50,0.371696,0.405983,0.110981
6,Two-Tower,10,0.002564,0.002564,0.000920
7,Two-Tower,20,0.002991,0.003419,0.001066
8,Two-Tower,50,0.003846,0.004274,0.001221


## Result interpretation and next boundary

本轮只验证现有 checkpoint 的真实 retrieval 能力，不改变训练方式或重新训练。若 Two-Tower 弱于 ItemCF，保留结果，并结合当前实现检查 PV→purchase 目标错位、uniform random negatives、ID-only 表达和仅训练 3 epochs 等已知限制。

FAISS IndexFlatIP 只作为 exact retrieval 的实现一致性检查；当前环境没有 FAISS，因此本轮不修改环境或引入近似检索。

In [6]:
import importlib.util
two_tower_r50 = float(two_tower_results.loc[two_tower_results['K'].eq(50), 'Recall'].iloc[0])
itemcf_r50 = float(itemcf_warm_results.loc[itemcf_warm_results['K'].eq(50), 'Recall'].iloc[0])
top1_items = [items[0] for items in two_tower_recommendations.values()]
top50_catalog = {item for items in two_tower_recommendations.values() for item in items}
top1_counts = pd.Series(top1_items).value_counts()
print('Two-Tower Recall@50:', two_tower_r50)
print('ItemCF Recall@50:', itemcf_r50)
print('Unique Top-1 items across evaluation users:', len(top1_counts))
print('Most common Top-1 item/share:', int(top1_counts.index[0]), float(top1_counts.iloc[0] / len(top1_items)))
print('Unique items exposed in all Top-50 lists:', len(top50_catalog), '/', recovered_items)
print('Two-Tower weaker than ItemCF:', two_tower_r50 < itemcf_r50)
if two_tower_r50 < itemcf_r50:
    print('Likely causes to test later: PV/purchase objective mismatch; uniform random negatives; ID-only towers; only 3 training epochs.')
print('FAISS available:', importlib.util.find_spec('faiss') is not None)
print('FAISS decision: skipped; exact chunked retrieval is the correctness baseline and the environment has no FAISS.')
print('NEXT_RECOMMENDED_STEP: fix retrieval-aware validation and negative sampling, then retrain a separately versioned checkpoint before Ranking.')

Two-Tower Recall@50: 0.0038461538461538464
ItemCF Recall@50: 0.37169580419580417
Unique Top-1 items across evaluation users: 2
Most common Top-1 item/share: 3845720 0.9196581196581196
Unique items exposed in all Top-50 lists: 301 / 302016
Two-Tower weaker than ItemCF: True
Likely causes to test later: PV/purchase objective mismatch; uniform random negatives; ID-only towers; only 3 training epochs.
FAISS available: False
FAISS decision: skipped; exact chunked retrieval is the correctness baseline and the environment has no FAISS.
NEXT_RECOMMENDED_STEP: fix retrieval-aware validation and negative sampling, then retrain a separately versioned checkpoint before Ranking.
